In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ============================================================================
# pipeline_final.ipynb  -  hallucination detect-and-correct pipeline,
#                          4-arm experiment on the full 140 round-2 questions
# CONSOLIDATED: every cell is at its latest version. Paste each CELL block into
# its own Colab cell, in order (CELL 0 .. CELL 12b). No cross-referencing needed.
#
# Relationship to pipeline.ipynb (reviewed20, FROZEN):
#   * Pipeline logic is IDENTICAL: same prompts, models, MAX_ITERS, TOP_K, SEED,
#     ARMS, verify-and-revise flow, shared init answer + shared decomposition
#     per question per run, same payload schema.
#   * Only infrastructure differs: question selection by RUN_SET from
#     dataset_verified_final.jsonl, per-question checkpointing, outputs to
#     RESULTS_FINAL. The round-1 20 questions are NEVER re-run; their frozen
#     results are reused (verified by CELL 2c) and merged in CELL 12b.
#
# Design summary (defensible points, unchanged from reviewed20):
#   * Self-sufficient: loads the corpus FROM DISK; no dependency on build_corpus's
#     kernel. A kernel restart does not force a rebuild.
#   * Verify-ONLY: the initial answer is generated WITHOUT retrieval; the guideline
#     is used only to CHECK claims, never to write them. (We study DETECTION.)
#   * Clean comparison: per question we generate ONE initial answer and ONE
#     decomposition; all arms verify the SAME claim set at iteration 1. Only the
#     retrieval (index + filter) differs across arms.
#   * FOUR arms:
#       coarse            - whole-record chunks, no filter (baseline)
#       atomic            - recommendation-atomic chunks, no filter
#       atomic_gl_filter  - atomic + restrict retrieval to the question's guideline
#       atomic_sec_filter - atomic + restrict retrieval to the question's section_group
#   * Rich recording: each verdict stores verdict, retrieved_ids (authoritative),
#     model-cited chunk_id (model-typed caveat), rationale, target class/level, and
#     audit slots (human_label, contradiction_kind, error_type).
#   * Detection vs hallucination: the pipeline flags CONTRADICTIONS against ESC;
#     the manual audit classifies each as true_hallucination / cross_source /
#     version_mismatch / verifier_error. We report hallucination-precision.
#
# Robustness (new):
#   * Per-question checkpointing: each finished question is appended to
#     ckpt_<run_tag>.jsonl immediately. A Colab disconnect loses at most the
#     question in flight; rerunning the same cell skips completed uids and
#     retries failed ones. The run file is written only when complete.
#   * Every run saved to RESULTS_FINAL, timestamped, overwrite-safe,
#     self-contained. Frozen reviewed20 files in RESULTS_DIR are never touched.
#
# RUN ORDER:
#   setup:   config.py updated (DATASET_REVIEWED20, DATASET_FINAL, RESULTS_FINAL)
#            run cells 0,1,2,2b,2c,3..8 defs, 9,10,11,12,12b
#   smoke:   RUN_SET="smoke3"; sample = build_sample()
#            payload_s, p_s = run_batch(sample, run_tag="smoke_test")
#            inspect output, then DELETE ckpt_smoke_test.jsonl and the smoke
#            result file (throwaway, must not contaminate real runs)
#   batches: for batch b in 1..4 (batch 1 first, contains the 12 re-sent):
#              RUN_SET=f"batch{b}"; sample = build_sample()
#              payload, p1 = run_batch(sample, run_tag="batch{b}_run1")
#                (if INCOMPLETE: rerun the same cell until SAVED)
#              payload, p2 = run_batch(sample, run_tag="batch{b}_run2")
#              payload, p3 = run_batch(sample, run_tag="batch{b}_run3")
#              combined_b, c_b = combine_runs(paths=[p1,p2,p3], out_tag=f"batch{b}")
#              summary_table / class_stratified / export_disagreements as before
#   merge:   combined_all140, c140 = merge_combined([c_b1,c_b2,c_b3,c_b4],
#                                                   out_tag="all140")
#            combined_all160, c160 = merge_combined(
#                [c140, RESULTS_DIR + "/combined_reviewed20.json"], out_tag="all160")
#   audit:   audit_path = export_disagreements(payload)   # fill the audit fields
#   metrics: detection_metrics(audit_path)
# ============================================================================

In [ ]:
# =========================== CELL 0 : setup =================================
!pip -q install openai sentence-transformers faiss-cpu
from google.colab import drive
drive.mount('/content/drive')

import os, json, re, random, time, sys, datetime, glob
from getpass import getpass
from openai import OpenAI

sys.path.insert(0, "/content/drive/MyDrive/medical_qa_hallucination/experiments")
from config import *   # ROOT, INDEX_DIR, RESULTS_DIR, RESULTS_FINAL,
                       # DATASET_FINAL, DATASET_REVIEWED20 (alias: DATASET),
                       # EMBED_MODEL, GEN_MODEL, VERIFY_MODEL,
                       # MAX_ITERS, TOP_K, SEED, ARMS

client = OpenAI(api_key=getpass("OpenAI API key: "))

TOKENS = {"in": 0, "out": 0}
def _track(resp):
    u = getattr(resp, "usage", None)
    if u:
        TOKENS["in"]  += getattr(u, "prompt_tokens", 0) or 0
        TOKENS["out"] += getattr(u, "completion_tokens", 0) or 0

def chat(model, system, user, temperature=None, max_tokens=1200, seed=SEED):
    kwargs = dict(model=model, max_completion_tokens=max_tokens, seed=seed,
                  messages=[{"role": "system", "content": system},
                            {"role": "user", "content": user}])
    if temperature is not None:
        kwargs["temperature"] = temperature
    import time
    for attempt in range(6):
        try:
            resp = client.chat.completions.create(**kwargs)
            _track(resp)
            return resp.choices[0].message.content.strip()
        except Exception as e:
            if "rate_limit" in str(e).lower() or "429" in str(e):
                wait = min(2 ** attempt, 30)
                print(f"   [chat] rate limited, waiting {wait}s (attempt {attempt+1})")
                time.sleep(wait)
                continue
            raise
    raise RuntimeError("chat: exhausted rate-limit retries")

print("client ready. models:", GEN_MODEL, "/", VERIFY_MODEL)
print("arms:", list(ARMS.keys()))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 74.2 MB/s eta 0:00:00
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
OpenAI API key: ··········
client ready. models: gpt-5.4 / gpt-4.1-mini
arms: ['coarse', 'atomic', 'atomic_gl_filter', 'atomic_sec_filter']


In [ ]:
# =========================== CELL 1 : self-sufficient corpus loader =========
import faiss, numpy as np
from sentence_transformers import SentenceTransformer

print("loading embedding model:", EMBED_MODEL)
MODEL = SentenceTransformer(EMBED_MODEL)

def _load_index(name):
    index = faiss.read_index(os.path.join(INDEX_DIR, f"{name}.faiss"))
    meta = [json.loads(l) for l in open(os.path.join(INDEX_DIR, f"{name}_meta.jsonl"),
                                        encoding="utf-8") if l.strip()]
    assert index.ntotal == len(meta), \
        f"{name}: faiss {index.ntotal} vs meta {len(meta)} - out of sync!"
    return {"index": index, "meta": meta}

INDEX_CACHE = {n: _load_index(n) for n in ("coarse", "atomic")}
for n, d in INDEX_CACHE.items():
    print(f"  {n}: {d['index'].ntotal} vectors, dim {d['index'].d}")

def _locator(m):
    return (m.get("table_id") or m.get("figure_id") or m.get("box_id")
            or m.get("section") or "?")

def retrieve(query, index_name, k=TOP_K, filters=None):
    """Search a cached index. Returns enriched dicts incl. stable position-based
    chunk_id and the chunk's class/level/section_group. Over-fetch + post-filter
    when `filters` is given (FAISS can't filter metadata itself)."""
    d = INDEX_CACHE[index_name]
    index, meta = d["index"], d["meta"]
    qv = np.asarray(MODEL.encode([query], normalize_embeddings=True), dtype="float32")
    fetch = k * 20 if filters else k
    scores, idxs = index.search(qv, min(fetch, len(meta)))
    out = []
    for score, i in zip(scores[0], idxs[0]):
        if i < 0:
            continue
        m = meta[i]
        if filters and any(m.get(fk) != fv for fk, fv in filters.items()):
            continue
        cid = f"{index_name}:{int(i):04d}:{m.get('code')}:{m.get('chunk_type')}:{_locator(m)}"
        out.append({
            "chunk_id": cid, "pos": int(i), "score": round(float(score), 3),
            "code": m.get("code"), "section_group": m.get("section_group"),
            "chunk_type": m.get("chunk_type"), "locator": _locator(m),
            "class": m.get("class"), "level": m.get("level"),
            "text": m.get("text", ""),
        })
        if len(out) >= k:
            break
    return out

print("retrieve() ready (self-sufficient, position-stable ids)")

loading embedding model: BAAI/bge-small-en-v1.5


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

... (5 outputs left out)

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

  coarse: 525 vectors, dim 384
  atomic: 1035 vectors, dim 384
retrieve() ready (self-sufficient, position-stable ids)


In [ ]:
# =========================== CELL 2 : run-set selection =====================
def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

def parse_uid(uid):
    """uid format {CODE}_S{SECTION}_Q{N}  e.g. CM_S6-10-6-11_Q1
       -> code='CM', section_group='CM_S6-10-6-11' (section uses hyphens, no '_')."""
    parts = uid.split("_")
    code = parts[0]
    section_group = "_".join(parts[:-1])   # everything except the trailing Q{n}
    return code, section_group

# ---- choose the run set -------------------------------------------------------
# RUN_SET selects which questions THIS session processes:
#   "batch1".."batch4" -> the 35 questions of that review batch (review_round 2)
#   "all140"           -> all 140 round-2 questions
#   "smoke3"           -> first 3 questions of batch1 (pipeline test, throwaway)
# NOTE: there is deliberately no "all160" RUN_SET. The round-1 20 are never
# re-run; their frozen results are reused (checked by CELL 2c, merged in 12b).
RUN_SET = "batch1"

def _question_text(r):
    """Prefer the doctor's corrected text when present."""
    return (r.get("final_question_text")
            or r.get("question_text")
            or "")

def build_sample(run_set=RUN_SET):
    rows = load_jsonl(DATASET_FINAL)
    assert len(rows) == 160, f"dataset_verified_final.jsonl: {len(rows)} rows != 160"

    if run_set.startswith("batch"):
        bno = int(run_set[-1])
        assert 1 <= bno <= 4, f"bad RUN_SET: {run_set}"
        sample = [r for r in rows
                  if r.get("review_round") == 2 and r.get("review_batch") == bno]
        assert len(sample) == 35, f"{run_set}: {len(sample)} rows != 35"
    elif run_set == "all140":
        sample = [r for r in rows if r.get("review_round") == 2]
        assert len(sample) == 140, f"all140: {len(sample)} rows != 140"
    elif run_set == "smoke3":
        sample = [r for r in rows
                  if r.get("review_round") == 2 and r.get("review_batch") == 1][:3]
        assert len(sample) == 3, f"smoke3: {len(sample)} rows != 3"
    else:
        raise ValueError(f"unknown RUN_SET: {run_set!r}")

    for r in sample:
        c, sg = parse_uid(r["uid"])
        r["_code"], r["_section_group"] = c, sg
    n_corr = sum(1 for r in sample if str(r.get("expert_corrected", "")).strip())
    print(f"RUN_SET={run_set}: {len(sample)} questions "
          f"({n_corr} with expert corrections)")
    for r in sample:
        print(f"   {r['uid']:24s} {r.get('question_type','?'):16s} "
              f"code={r['_code']:4s} sg={r['_section_group']}")
    return sample

sample = build_sample()

RUN_SET=batch1: 35 questions (7 with expert corrections)
   VHD_S4_Q2                fact_based       code=VHD  sg=VHD_S4
   VHD_S5_Q2                fact_based       code=VHD  sg=VHD_S5
   VHD_S5_Q4                fact_based       code=VHD  sg=VHD_S5
   VHD_S6_Q6                fact_based       code=VHD  sg=VHD_S6
   VHD_S7_Q3                patient_vignette code=VHD  sg=VHD_S7
   VHD_S7_Q8                fact_based       code=VHD  sg=VHD_S7
   VHD_S10_Q1               patient_vignette code=VHD  sg=VHD_S10
   VHD_S10_Q2               patient_vignette code=VHD  sg=VHD_S10
   VHD_S10_Q4               patient_vignette code=VHD  sg=VHD_S10
   VHD_S11_Q7               patient_vignette code=VHD  sg=VHD_S11
   VHD_S11_Q8               fact_based       code=VHD  sg=VHD_S11
   VHD_S13_Q2               patient_vignette code=VHD  sg=VHD_S13
   VHD_S13_Q4               patient_vignette code=VHD  sg=VHD_S13
   VHD_S13_Q5               patient_vignette code=VHD  sg=VHD_S13
   VHD_S14_Q1            

In [ ]:
# =========================== CELL 2b : pre-flight check (optional) ===========
# Verify every sampled question's section_group exists in the atomic meta, so the
# section-filter arm cannot silently return empty. Run once after build + sample.
def preflight_section_groups():
    atomic_sgs = {m.get("section_group") for m in INDEX_CACHE["atomic"]["meta"]}
    ok = True
    for r in sample:
        if r["_section_group"] not in atomic_sgs:
            ok = False
            print(f"  [!] {r['uid']}: section_group '{r['_section_group']}' NOT in atomic meta")
    if ok:
        print(f"pre-flight OK: all {len(sample)} section_groups present in atomic meta.")
    else:
        print("pre-flight FAILED: fix section_group derivation / uid mapping.")
    return ok

preflight_section_groups()

pre-flight OK: all 35 section_groups present in atomic meta.


True

In [ ]:
# ================= CELL 2c : frozen-20 reuse integrity check =================
def check_frozen20_consistency():
    """The reviewed20 results are reused as-is. Verify the question text they
    were generated from is identical to what dataset_verified_final holds for
    those uids (it must be: round-1 rows were carried over untouched)."""
    rows = load_jsonl(DATASET_FINAL)
    final20 = {r["uid"]: _question_text(r)
               for r in rows if r.get("review_round") == 1}
    assert len(final20) == 20, f"round-1 rows in final dataset: {len(final20)} != 20"

    cand = os.path.join(RESULTS_DIR, "results_reviewed20_run1.json")
    if not os.path.exists(cand):
        hits = sorted(glob.glob(os.path.join(RESULTS_DIR, "results_reviewed20_run1*.json")))
        assert hits, "no frozen reviewed20 run1 file found"
        cand = hits[-1]
    frozen = json.load(open(cand, encoding="utf-8"))
    assert len(frozen["uids"]) == 20, f"frozen run1: {len(frozen['uids'])} != 20"

    bad = [res["uid"] for res in frozen["results"]
           if res["question"] != final20.get(res["uid"])]
    assert not bad, f"question text mismatch for: {bad}"
    print("frozen-20 reuse OK: question texts identical to", os.path.basename(cand))

check_frozen20_consistency()

frozen-20 reuse OK: question texts identical to results_reviewed20_run1.json


In [ ]:
# =========================== CELL 3 : Stage 1 generate (+ Stage 5 rewrite) ==
GEN_SYS = ("You are a cardiology expert answering a clinical question using ESC "
           "guideline knowledge. Write a focused answer of 6-7 sentences. Be "
           "specific about thresholds, classes, and criteria where relevant. Do "
           "not hedge or add disclaimers.")

def generate_answer(question, feedback=None):
    user = f"Question:\n{question}\n\nWrite the answer."
    if feedback:
        user += ("\n\nA previous draft contained unsupported claims. Revise to fix "
                 "these specific problems, keeping everything else:\n" + feedback)
    return chat(GEN_MODEL, GEN_SYS, user, max_tokens=600)

print("generate_answer() ready")

generate_answer() ready


In [ ]:
# =========================== CELL 4 : Stage 2 decompose (shared) ============
DECOMP_SYS = (
    "Decompose the answer into atomic factual claims. Each claim states ONE fact. "
    "Tag each claim:\n"
    "  type='guideline_fact'  -> a general medical/guideline statement (thresholds, "
    "classes, drugs, indications) that could be checked against a guideline.\n"
    "  type='patient_specific' -> a statement about THIS patient's data or an "
    "inference about this individual; NOT checkable against a general guideline.\n"
    "Return ONLY a JSON array, no prose, no markdown fences. Each element: "
    '{"id": int, "claim": str, "type": "guideline_fact"|"patient_specific"}.')

def _parse_json(txt):
    txt = re.sub(r"^```(json)?|```$", "", txt.strip(), flags=re.M).strip()
    if not txt.startswith("["):
        a, b = txt.find("["), txt.rfind("]")
        if a != -1 and b != -1 and b > a:
            txt = txt[a:b+1]
    return json.loads(txt)

def _salvage_json_array(txt):
    """Recover as many complete {...} objects as possible from a (possibly
    TRUNCATED) JSON array. Used when the model's JSON was cut off mid-string by
    the token cap. Scans for balanced top-level objects and parses each."""
    txt = re.sub(r"^```(json)?|```$", "", txt.strip(), flags=re.M).strip()
    objs, depth, start, in_str, esc = [], 0, None, False, False
    for i, ch in enumerate(txt):
        if in_str:
            if esc:        esc = False
            elif ch == "\\": esc = True
            elif ch == '"':  in_str = False
            continue
        if ch == '"':
            in_str = True
        elif ch == "{":
            if depth == 0:
                start = i
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0 and start is not None:
                frag = txt[start:i+1]
                try:
                    objs.append(json.loads(frag))
                except Exception:
                    pass
                start = None
    return objs

def decompose(answer):
    """Return a list of claim dicts. Never raises. Token cap is tiered:
    try 1000 (covers the vast majority cheaply), escalate to 1500 only if the
    output truncated/failed, then salvage complete objects, worst case []."""
    # attempt 1: default cap
    out1 = chat(VERIFY_MODEL, DECOMP_SYS, f"Answer:\n{answer}",
                temperature=0.0, max_tokens=1000)
    try:
        return _parse_json(out1)
    except Exception as e:
        print("   [decompose] parse failed at 1000 tokens, retrying with 1500:", e)
    # attempt 2: escalate cap + explicit nudge (for genuinely long answers)
    out2 = chat(VERIFY_MODEL, DECOMP_SYS,
                f"Answer:\n{answer}\n\nReturn ONLY valid JSON.",
                temperature=0.0, max_tokens=1500)
    try:
        return _parse_json(out2)
    except Exception as e2:
        # last resort: salvage complete objects from whichever output is longer
        salv = _salvage_json_array(out2) or _salvage_json_array(out1)
        if salv:
            print(f"   [decompose] salvaged {len(salv)} claims from truncated JSON")
            return salv
        print("   [decompose] UNRECOVERABLE, returning [] (question will be skipped):", e2)
        return []

print("decompose() ready")

decompose() ready


In [ ]:
# =========================== CELL 5 : Stage 3+4 retrieve + verify ===========
VERIFY_SYS = (
    "You verify a single clinical claim against retrieved ESC guideline passages. "
    "Use ONLY the passages provided. Decide:\n"
    "  'supported'    -> a passage clearly supports the claim.\n"
    "  'refuted'      -> a passage clearly contradicts the claim.\n"
    "  'unverifiable' -> the passages do not contain enough to judge.\n"
    "Return ONLY JSON, no fences: "
    '{"verdict": str, "chunk_id": str|null, "rationale": str}. '
    "chunk_id is the id of the single most relevant passage (or null if none). "
    "rationale is ONE short sentence.")

def _format_passages(hits):
    return "\n\n".join(f"[{h['chunk_id']}]\n{h['text']}" for h in hits)

def _target_grade(hits):
    for h in hits:
        if h.get("class") or h.get("level"):
            return h.get("class"), h.get("level")
    return None, None

def verify_claim(claim_text, index_name, filters=None):
    hits = retrieve(claim_text, index_name, k=TOP_K, filters=filters)
    user = (f"Claim:\n{claim_text}\n\nRetrieved passages:\n"
            f"{_format_passages(hits)}\n\nVerify the claim.")
    out = chat(VERIFY_MODEL, VERIFY_SYS, user, temperature=0.0, max_tokens=300)
    try:
        v = _parse_json(out)
    except Exception:
        v = {"verdict": "unverifiable", "chunk_id": None, "rationale": "parse_error"}
    tc, tl = _target_grade(hits)
    v["retrieved_ids"] = [h["chunk_id"] for h in hits]   # AUTHORITATIVE
    v["target_class"]  = tc
    v["target_level"]  = tl
    v["human_label"]        = None
    v["contradiction_kind"] = None
    v["error_type"]         = None
    return v

def verify_answer(claims, index_name, filters=None):
    """Verify guideline_fact claims; patient_specific -> auto unverifiable."""
    results = []
    for c in claims:
        if c.get("type") == "patient_specific":
            results.append({**c, "verdict": "unverifiable", "chunk_id": None,
                            "rationale": "patient-specific; not a guideline fact",
                            "retrieved_ids": [], "target_class": None,
                            "target_level": None, "human_label": None,
                            "contradiction_kind": None, "error_type": None})
        else:
            results.append({**c, **verify_claim(c["claim"], index_name, filters)})
    return results

print("verify_answer() ready")

verify_answer() ready


In [ ]:
# =========================== CELL 6 : Stage 5 rewrite loop (one arm) =========
def refuted_feedback(verified):
    bad = [r for r in verified if r["verdict"] == "refuted"]
    if not bad:
        return None
    return "\n".join(f"- Claim '{r['claim']}' was refuted: {r.get('rationale','')}"
                     for r in bad)

def _filters_for(arm_name, qcode, qgroup):
    _, mode = ARMS[arm_name]
    if mode == "guideline":
        return {"code": qcode}
    if mode == "section_group":
        return {"section_group": qgroup}
    return None

def _force_guideline_fact(claims, qtype):
    """Fact-based questions are built purely from guideline content (no patient
    data), so every claim must be a guideline_fact. Override any patient_specific
    tag the decomposer may wrongly assign, so no real error is hidden as
    auto-unverifiable. Vignette questions keep the decomposer's tags."""
    if qtype == "fact_based":
        for c in claims:
            c["type"] = "guideline_fact"
    return claims

def run_arm(question, arm_name, init_answer, init_claims, qcode, qgroup,
            qtype=None, verbose=True):
    """Full verify-and-revise loop for ONE question under ONE arm.
    Iteration 1 uses the SHARED init_answer + SHARED init_claims; later iterations
    re-decompose this arm's own rewritten answer. For fact_based questions all
    claims are forced to guideline_fact (no patient data by construction)."""
    index_name, _ = ARMS[arm_name]
    filters = _filters_for(arm_name, qcode, qgroup)
    answer = init_answer
    history = []
    for it in range(1, MAX_ITERS + 1):
        if it == 1:
            claims = init_claims
        else:
            claims = _force_guideline_fact(decompose(answer), qtype)
        verified = verify_answer(claims, index_name, filters)
        n_ref = sum(1 for r in verified if r["verdict"] == "refuted")
        n_sup = sum(1 for r in verified if r["verdict"] == "supported")
        n_unv = sum(1 for r in verified if r["verdict"] == "unverifiable")
        history.append({"iter": it, "answer": answer, "verified": verified,
                        "n_claims": len(claims), "supported": n_sup,
                        "refuted": n_ref, "unverifiable": n_unv})
        if verbose:
            print(f"     [{arm_name:17s}] iter {it}: {len(claims):2d} claims | "
                  f"sup {n_sup:2d}  ref {n_ref:2d}  unv {n_unv:2d}")
        fb = refuted_feedback(verified)
        if not fb:
            break
        if it < MAX_ITERS:
            answer = generate_answer(question, feedback=fb)
    return {"arm": arm_name, "index": index_name, "filter": ARMS[arm_name][1],
            "final_answer": answer, "iterations": len(history),
            "history": history, "final_refuted": history[-1]["refuted"],
            "ref_init": history[0]["refuted"]}

print("run_arm() ready")

run_arm() ready


In [ ]:
# =========================== CELL 7 : one question, ALL arms ================
def run_question(row, verbose=True):
    q     = _question_text(row)      # prefers final_question_text (doctor's correction)
    qcode = row["_code"]
    qgrp  = row["_section_group"]
    qtype = row.get("question_type")
    if verbose:
        print("=" * 74)
        print("UID:", row["uid"], "|", qtype, "| code", qcode, "| sg", qgrp)
    init_answer = generate_answer(q)
    init_claims = _force_guideline_fact(decompose(init_answer), qtype)
    if verbose:
        n_pat = sum(1 for c in init_claims if c.get("type") == "patient_specific")
        print(f"   shared init: {len(init_claims)} claims "
              f"({n_pat} patient_specific)")
    arms_out = {}
    for arm_name in ARMS:
        arms_out[arm_name] = run_arm(q, arm_name, init_answer, init_claims,
                                     qcode, qgrp, qtype=qtype, verbose=verbose)
    return {"uid": row["uid"], "type": row.get("question_type"),
            "code": qcode, "section_group": qgrp,
            "question": q, "init_answer": init_answer,
            "init_claims": init_claims, "arms": arms_out}

# smoke test (optional): demo = run_question(sample[0])

In [ ]:
# ================= CELL 8 : checkpointed batch runner + safe save ===========
def _unique_path(path):
    """If `path` exists, insert _v2, _v3, ... before the extension (no clobber)."""
    if not os.path.exists(path):
        return path
    base, ext = os.path.splitext(path); k = 2
    while os.path.exists(f"{base}_v{k}{ext}"):
        k += 1
    return f"{base}_v{k}{ext}"

def _ckpt_path(run_tag):
    return os.path.join(RESULTS_FINAL, f"ckpt_{run_tag}.jsonl")

def _load_ckpt(run_tag):
    path = _ckpt_path(run_tag)
    recs = {}
    if os.path.exists(path):
        with open(path, encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    r = json.loads(line)
                    recs[r["uid"]] = r
    return recs

def _append_ckpt(run_tag, record):
    with open(_ckpt_path(run_tag), "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

def run_batch(rows, run_tag, verbose=True):
    """Process `rows` under `run_tag` with per-question checkpointing.
    Rerunning this cell is always safe: checkpointed uids are skipped, failed
    uids are retried. When the checkpoint covers all rows, assemble the
    standard payload (schema identical to the reviewed20 runs) and save."""
    expected_uids = [r["uid"] for r in rows]
    done = _load_ckpt(run_tag)
    extra = set(done) - set(expected_uids)
    assert not extra, \
        f"ckpt_{run_tag}.jsonl has uids outside this RUN_SET: {sorted(extra)[:5]}"

    todo = [r for r in rows if r["uid"] not in done]
    print(f"{run_tag}: {len(done)}/{len(rows)} already checkpointed, "
          f"{len(todo)} to go this session")

    tok_before = dict(TOKENS)
    failed = []
    for i, row in enumerate(todo, 1):
        print(f"\n[{i}/{len(todo)}] {row['uid']}")
        try:
            rec = run_question(row, verbose=verbose)
            _append_ckpt(run_tag, rec)
            done[rec["uid"]] = rec
        except Exception as e:
            msg = f"{type(e).__name__}: {e}"
            # one bad question must never lose the run; it stays
            # un-checkpointed and is retried on the next execution of this cell
            print(f"   [!] {row['uid']} FAILED ({msg}); "
                  f"will retry on next execution of this cell.")
            failed.append({"uid": row["uid"], "error": msg})
            if "insufficient_quota" in str(e):
                print("   [!] CREDITS EXHAUSTED - stopping this session. "
                      "Top up, then rerun this cell; completed questions "
                      "are safe in the checkpoint.")
                break

    if len(done) < len(rows):
        print(f"\nINCOMPLETE: {len(done)}/{len(rows)} done "
              f"({len(failed)} failed this session). Rerun this cell to continue; "
              f"checkpoint: {_ckpt_path(run_tag)}")
        return None, _ckpt_path(run_tag)

    run_tokens = {"in":  TOKENS["in"]  - tok_before["in"],
                  "out": TOKENS["out"] - tok_before["out"]}
    records = [done[u] for u in expected_uids]
    payload = {
        "run_tag": run_tag,
        "run_set": RUN_SET,
        "timestamp": datetime.datetime.now().isoformat(timespec="seconds"),
        "n_questions": len(records),
        "n_failed": 0,          # all questions present at finalize time
        "failed": [],
        "arms": list(ARMS.keys()),
        "config": {"GEN_MODEL": GEN_MODEL, "VERIFY_MODEL": VERIFY_MODEL,
                   "MAX_ITERS": MAX_ITERS, "TOP_K": TOP_K, "SEED": SEED,
                   "EMBED_MODEL": EMBED_MODEL},
        "uids": expected_uids,
        "tokens_this_run": run_tokens,   # NOTE: final session only (resume)
        "tokens_cumulative": dict(TOKENS),
        "results": records,
    }
    stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    path = _unique_path(os.path.join(RESULTS_FINAL, f"results_{run_tag}_{stamp}.json"))
    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)
    print(f"\nSAVED -> {path}   ({len(records)} questions)")
    print(f"this session tokens: in={run_tokens['in']} out={run_tokens['out']}  "
          f"| cumulative this kernel: in={TOKENS['in']} out={TOKENS['out']}")
    return payload, path

In [ ]:
# =========================== CELL 9 : summary tables ========================
def summary_table(payload):
    arms = payload["arms"]
    hdr = f"{'uid':24s} {'type':10s}" + "".join(f" {a[:10]:>11s}" for a in arms)
    print("=" * len(hdr)); print(hdr); print("-" * len(hdr))
    totals = {a: 0 for a in arms}
    for r in payload["results"]:
        line = f"{r['uid']:24s} {str(r['type'])[:10]:10s}"
        for a in arms:
            ri = r["arms"][a]["ref_init"]; totals[a] += ri
            line += f" {ri:>11d}"
        print(line)
    print("-" * len(hdr))
    tline = f"{'FIRST-PASS REFUTATIONS':24s} {'':10s}"
    for a in arms:
        tline += f" {totals[a]:>11d}"
    print(tline)
    return totals

def class_stratified(payload):
    arms = payload["arms"]
    table = {a: {} for a in arms}
    for r in payload["results"]:
        for a in arms:
            for v in r["arms"][a]["history"][0]["verified"]:
                if v["verdict"] == "refuted":
                    cls = v.get("target_class") or "none"
                    table[a][cls] = table[a].get(cls, 0) + 1
    classes = sorted({c for a in arms for c in table[a]})
    print("\nrefutations by target class of recommendation:")
    hdr = f"{'class':8s}" + "".join(f" {a[:10]:>11s}" for a in arms)
    print(hdr); print("-" * len(hdr))
    for c in classes:
        print(f"{c:8s}" + "".join(f" {table[a].get(c,0):>11d}" for a in arms))

In [ ]:
# =========================== CELL 10 : audit export =========================
def export_disagreements(payload, out_name=None):
    """Dump every first-pass refutation into an audit file. Fill per row:
      human_label        : "TP" | "FP"
      contradiction_kind : "true_hallucination" | "cross_source"
                           | "version_mismatch" | "verifier_error" | "other"
      error_type         : "numeric" | "class" | "level" | "entity" | "other"
    Then run detection_metrics(audit_path)."""
    rows = []
    for r in payload["results"]:
        if not any(r["arms"][a]["ref_init"] > 0 for a in payload["arms"]):
            continue
        for a in payload["arms"]:
            for v in r["arms"][a]["history"][0]["verified"]:
                if v["verdict"] == "refuted":
                    rows.append({
                        "uid": r["uid"], "arm": a,
                        "claim": v["claim"], "verdict": v["verdict"],
                        "model_chunk_id": v.get("chunk_id"),
                        "retrieved_ids": v.get("retrieved_ids"),
                        "rationale": v.get("rationale"),
                        "target_class": v.get("target_class"),
                        "target_level": v.get("target_level"),
                        "human_label": None,
                        "contradiction_kind": None,
                        "error_type": None,
                        "audit_note": "",
                    })
    out_name = out_name or f"audit_{payload['run_tag']}.json"
    path = _unique_path(os.path.join(RESULTS_FINAL, out_name))
    with open(path, "w", encoding="utf-8") as f:
        json.dump(rows, f, ensure_ascii=False, indent=2)
    print(f"exported {len(rows)} refuted claims to audit -> {path}")
    print("Fill human_label, contradiction_kind, error_type per row, then run detection_metrics().")
    return path

In [ ]:
# =========================== CELL 11 : detection metrics ====================
def detection_metrics(audit_path):
    rows = json.load(open(audit_path, encoding="utf-8"))
    arms = sorted({r["arm"] for r in rows})
    KINDS  = ["true_hallucination", "cross_source", "version_mismatch",
              "verifier_error", "other", None]
    ETYPES = ["numeric", "class", "level", "entity", "other", None]

    print("=" * 64)
    print("PER-ARM: contradictions flagged vs TRUE hallucinations")
    print(f"{'arm':18s} {'flagged':>8s} {'true_hall':>10s} {'hall_precision':>15s}")
    print("-" * 64)
    for a in arms:
        sub = [r for r in rows if r["arm"] == a]
        flagged = len(sub)
        true_h  = sum(1 for r in sub if r["contradiction_kind"] == "true_hallucination")
        prec = true_h / flagged if flagged else float("nan")
        print(f"{a:18s} {flagged:>8d} {true_h:>10d} {prec:>15.2f}")

    print("\n" + "=" * 64)
    print("contradiction_kind breakdown (per arm):")
    hdr = f"{'arm':18s}" + "".join(f"{(k or 'none')[:12]:>13s}" for k in KINDS)
    print(hdr); print("-" * len(hdr))
    for a in arms:
        sub = [r for r in rows if r["arm"] == a]
        print(f"{a:18s}" + "".join(
            f"{sum(1 for r in sub if r['contradiction_kind']==k):>13d}" for k in KINDS))

    print("\n" + "=" * 64)
    print("error_type breakdown of TRUE hallucinations (per arm):")
    hdr = f"{'arm':18s}" + "".join(f"{(e or 'none')[:8]:>9s}" for e in ETYPES)
    print(hdr); print("-" * len(hdr))
    for a in arms:
        sub = [r for r in rows if r["arm"] == a
               and r["contradiction_kind"] == "true_hallucination"]
        print(f"{a:18s}" + "".join(
            f"{sum(1 for r in sub if r['error_type']==e):>9d}" for e in ETYPES))

    print("\nNotes: 'flagged' = contradictions vs ESC; 'true_hall' = audit-confirmed")
    print("hallucinations; hall_precision = true_hall/flagged. cross_source/")
    print("version_mismatch = defensible (NOT hallucinations); verifier_error = checker")
    print("misjudged. Sensitivity needs FN labels (audit a sample of supported claims).")

In [ ]:
# =========================== CELL 12 : combine separate runs ================
def combine_runs(paths=None, glob_tag=None, out_tag="combined", verbose=True):
    """Combine separate per-run result files into ONE summary file.
    Originals are never modified or deleted.
      paths    : explicit list of results_*.json (preferred)
      glob_tag : else load results_<glob_tag>*.json from RESULTS_FINAL"""
    if paths is None:
        assert glob_tag is not None, "pass paths=[...] or glob_tag=..."
        paths = sorted(glob.glob(os.path.join(RESULTS_FINAL, f"results_{glob_tag}*.json")))
    assert paths, "no run files found to combine"

    loaded = []
    for p in paths:
        with open(p, encoding="utf-8") as f:
            loaded.append((p, json.load(f)))
    if verbose:
        print(f"combining {len(loaded)} run files:")
        for p, _ in loaded:
            print("   ", os.path.basename(p))

    if len({tuple(sorted(d["uids"])) for _, d in loaded}) != 1:
        print("   [!] WARNING: runs do NOT share the same uid set.")

    arms = loaded[0][1]["arms"]
    per_run_totals = []
    for p, d in loaded:
        totals = {a: 0 for a in arms}
        for res in d["results"]:
            for a in arms:
                totals[a] += res["arms"][a]["ref_init"]
        per_run_totals.append({"file": os.path.basename(p), "run_tag": d.get("run_tag"),
                               "timestamp": d.get("timestamp"), "totals": totals})

    uids = loaded[0][1]["uids"]
    per_question = {u: {a: [] for a in arms} for u in uids}
    for _, d in loaded:
        by_uid = {res["uid"]: res for res in d["results"]}
        for u in uids:
            res = by_uid.get(u)
            for a in arms:
                per_question[u][a].append(res["arms"][a]["ref_init"] if res else None)

    def _mean(xs): return sum(xs)/len(xs) if xs else None
    agg = {a: {"per_run": [t["totals"][a] for t in per_run_totals]} for a in arms}
    for a in arms:
        xs = agg[a]["per_run"]
        agg[a].update({"mean": _mean(xs), "min": min(xs), "max": max(xs)})

    combined = {
        "out_tag": out_tag,
        "timestamp": datetime.datetime.now().isoformat(timespec="seconds"),
        "n_runs": len(loaded), "arms": arms, "uids": uids,
        "source_files": [os.path.basename(p) for p, _ in loaded],
        "per_run_totals": per_run_totals,
        "per_question_ref_init": per_question,
        "arm_aggregate": agg,
    }
    stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    cpath = _unique_path(os.path.join(RESULTS_FINAL, f"combined_{out_tag}_{stamp}.json"))
    with open(cpath, "w", encoding="utf-8") as f:
        json.dump(combined, f, ensure_ascii=False, indent=2)
    print(f"\nSAVED COMBINED -> {cpath}   (originals untouched)")

    print("\nfirst-pass refutations across runs (per arm):")
    hdr = f"{'arm':18s} " + "".join(f"{'r'+str(i+1):>5s}" for i in range(len(loaded))) \
          + f" {'mean':>7s} {'range':>9s}"
    print(hdr); print("-" * len(hdr))
    for a in arms:
        xs = agg[a]["per_run"]
        rng_str = f"{agg[a]['min']}-{agg[a]['max']}"
        print(f"{a:18s} " + "".join(f"{x:>5d}" for x in xs)
              + f" {agg[a]['mean']:>7.1f} {rng_str:>9s}")
    print("\nconsistency vs coarse (True = arm >= coarse that run):")
    for a in arms:
        if a == "coarse":
            continue
        flags  = [t["totals"][a] >= t["totals"]["coarse"] for t in per_run_totals]
        strict = [t["totals"][a] >  t["totals"]["coarse"] for t in per_run_totals]
        print(f"  {a:18s} >= coarse: {flags}   strictly >: {strict}")
    return combined, cpath

In [ ]:
# ================= CELL 12b : merge combined files (disjoint uids) ==========
def merge_combined(paths, out_tag="all140", verbose=True):
    """Merge combined_*.json files whose uid sets are DISJOINT into one summary:
    the four per-batch combineds -> all140, then all140 + frozen reviewed20
    -> all160. Originals are never modified."""
    loaded = []
    for p in paths:
        with open(p, encoding="utf-8") as f:
            loaded.append((p, json.load(f)))
    assert loaded, "no combined files given"
    arms = loaded[0][1]["arms"]
    assert all(d["arms"] == arms for _, d in loaded), "arm sets differ"

    all_uids = [u for _, d in loaded for u in d["uids"]]
    assert len(all_uids) == len(set(all_uids)), "uid sets are not disjoint"

    per_q, per_run_totals = {}, []
    for _, d in loaded:
        per_q.update(d["per_question_ref_init"])
        per_run_totals.extend(d["per_run_totals"])

    agg = {a: {"per_run": [t["totals"][a] for t in per_run_totals]} for a in arms}
    for a in arms:
        xs = agg[a]["per_run"]
        agg[a].update({"mean": sum(xs) / len(xs), "min": min(xs), "max": max(xs)})

    combined = {
        "out_tag": out_tag,
        "timestamp": datetime.datetime.now().isoformat(timespec="seconds"),
        "n_runs": len(per_run_totals), "arms": arms, "uids": all_uids,
        "source_files": [os.path.basename(p) for p, _ in loaded],
        "per_run_totals": per_run_totals,
        "per_question_ref_init": per_q,
        "arm_aggregate": agg,
    }
    stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    cpath = _unique_path(os.path.join(RESULTS_FINAL, f"combined_{out_tag}_{stamp}.json"))
    with open(cpath, "w", encoding="utf-8") as f:
        json.dump(combined, f, ensure_ascii=False, indent=2)
    print(f"SAVED MERGED -> {cpath}  ({len(all_uids)} questions, {len(per_run_totals)} runs)")
    print(f"\n{'arm':18s} {'mean':>7s} {'range':>8s}")
    for a in arms:
        print(f"{a:18s} {agg[a]['mean']:>7.1f} "
              f"{str(agg[a]['min']) + '-' + str(agg[a]['max']):>8s}")
    return combined, cpath

In [ ]:
# =========================== CELL 13 : smoke test (throwaway) ===============
RUN_SET = "smoke3"
sample = build_sample(RUN_SET)
preflight_section_groups()

payload_s, p_s = run_batch(sample, run_tag="smoke_test")
assert payload_s is not None, "smoke incomplete - rerun this cell"
summary_table(payload_s)

# eyeball a few verified claims in payload_s["results"], check costs on the
# OpenAI usage dashboard, then DELETE the smoke artifacts so they can never
# contaminate the real runs:
import os
os.remove(_ckpt_path("smoke_test"))
os.remove(p_s)
print("smoke artifacts deleted; set batch1 in CELL 14 and start the real runs")

RUN_SET=smoke3: 3 questions (0 with expert corrections)
   VHD_S4_Q2                fact_based       code=VHD  sg=VHD_S4
   VHD_S5_Q2                fact_based       code=VHD  sg=VHD_S5
   VHD_S5_Q4                fact_based       code=VHD  sg=VHD_S5
pre-flight OK: all 3 section_groups present in atomic meta.
smoke_test: 0/3 already checkpointed, 3 to go this session

[1/3] VHD_S4_Q2
UID: VHD_S4_Q2 | fact_based | code VHD | sg VHD_S4
   shared init: 12 claims (0 patient_specific)
     [coarse           ] iter 1: 12 claims | sup 11  ref  0  unv  1
     [atomic           ] iter 1: 12 claims | sup 11  ref  0  unv  1
     [atomic_gl_filter ] iter 1: 12 claims | sup 12  ref  0  unv  0
     [atomic_sec_filter] iter 1: 12 claims | sup 11  ref  0  unv  1

[2/3] VHD_S5_Q2
UID: VHD_S5_Q2 | fact_based | code VHD | sg VHD_S5
   shared init: 15 claims (0 patient_specific)
     [coarse           ] iter 1: 15 claims | sup 12  ref  0  unv  3
     [atomic           ] iter 1: 15 claims | sup 13  ref  0 

In [ ]:
# ==================== CELL 14 : batch execution helpers =====================
def do_batch_runs(run_set):
    """3 runs for one batch. Resumable: rerun-safe via checkpointing."""
    global RUN_SET, sample
    RUN_SET = run_set
    sample = build_sample(run_set)
    preflight_section_groups()
    run_paths = {}
    for r in (1, 2, 3):
        tag = f"{run_set}_run{r}"
        payload, run_paths[tag] = run_batch(sample, run_tag=tag)
        assert payload is not None, f"{tag} INCOMPLETE - fix, then rerun this cell"
        summary_table(payload)
        class_stratified(payload)
    print(f"\n{run_set} complete:", run_paths)
    return run_paths

def do_batch_combine(run_set):
    """Combine the batch's 3 runs -> combined_<batch>_<stamp>.json"""
    paths = sorted(glob.glob(os.path.join(RESULTS_FINAL,
                                          f"results_{run_set}_run*_*.json")))
    assert len(paths) == 3, \
        f"expected 3 run files for {run_set}, found {len(paths)}:\n" + "\n".join(paths)
    return combine_runs(paths=paths, out_tag=run_set)

In [ ]:
# CELL 14a : batch 1 (contains the 12 re-sent questions)
paths_b1 = do_batch_runs("batch1")
combined_b1, c_b1 = do_batch_combine("batch1")

RUN_SET=batch1: 35 questions (7 with expert corrections)
   VHD_S4_Q2                fact_based       code=VHD  sg=VHD_S4
   VHD_S5_Q2                fact_based       code=VHD  sg=VHD_S5
   VHD_S5_Q4                fact_based       code=VHD  sg=VHD_S5
   VHD_S6_Q6                fact_based       code=VHD  sg=VHD_S6
   VHD_S7_Q3                patient_vignette code=VHD  sg=VHD_S7
   VHD_S7_Q8                fact_based       code=VHD  sg=VHD_S7
   VHD_S10_Q1               patient_vignette code=VHD  sg=VHD_S10
   VHD_S10_Q2               patient_vignette code=VHD  sg=VHD_S10
   VHD_S10_Q4               patient_vignette code=VHD  sg=VHD_S10
   VHD_S11_Q7               patient_vignette code=VHD  sg=VHD_S11
   VHD_S11_Q8               fact_based       code=VHD  sg=VHD_S11
   VHD_S13_Q2               patient_vignette code=VHD  sg=VHD_S13
   VHD_S13_Q4               patient_vignette code=VHD  sg=VHD_S13
   VHD_S13_Q5               patient_vignette code=VHD  sg=VHD_S13
   VHD_S14_Q1            

In [ ]:
# CELL 14b : batch 2
paths_b2 = do_batch_runs("batch2")
combined_b2, c_b2 = do_batch_combine("batch2")

RUN_SET=batch2: 35 questions (12 with expert corrections)
   VHD_S5_Q3                fact_based       code=VHD  sg=VHD_S5
   VHD_S6_Q4                fact_based       code=VHD  sg=VHD_S6
   VHD_S6_Q7                fact_based       code=VHD  sg=VHD_S6
   VHD_S8_Q6                patient_vignette code=VHD  sg=VHD_S8
   VHD_S8_Q8                patient_vignette code=VHD  sg=VHD_S8
   VHD_S8_Q10               fact_based       code=VHD  sg=VHD_S8
   VHD_S8_Q11               fact_based       code=VHD  sg=VHD_S8
   VHD_S9_Q2                patient_vignette code=VHD  sg=VHD_S9
   VHD_S9_Q10               patient_vignette code=VHD  sg=VHD_S9
   VHD_S9_Q11               patient_vignette code=VHD  sg=VHD_S9
   VHD_S9_Q12               fact_based       code=VHD  sg=VHD_S9
   VHD_S10_Q3               patient_vignette code=VHD  sg=VHD_S10
   VHD_S11_Q3               patient_vignette code=VHD  sg=VHD_S11
   VHD_S13_Q1               patient_vignette code=VHD  sg=VHD_S13
   VHD_S13_Q6               p

In [ ]:
# CELL 14c : batch 3
paths_b3 = do_batch_runs("batch3")
combined_b3, c_b3 = do_batch_combine("batch3")

RUN_SET=batch3: 35 questions (6 with expert corrections)
   VHD_S5_Q1                fact_based       code=VHD  sg=VHD_S5
   VHD_S5_Q5                fact_based       code=VHD  sg=VHD_S5
   VHD_S6_Q3                fact_based       code=VHD  sg=VHD_S6
   VHD_S7_Q4                patient_vignette code=VHD  sg=VHD_S7
   VHD_S7_Q5                patient_vignette code=VHD  sg=VHD_S7
   VHD_S8_Q12               fact_based       code=VHD  sg=VHD_S8
   VHD_S9_Q4                patient_vignette code=VHD  sg=VHD_S9
   VHD_S9_Q9                patient_vignette code=VHD  sg=VHD_S9
   VHD_S9_Q13               fact_based       code=VHD  sg=VHD_S9
   VHD_S10_Q5               fact_based       code=VHD  sg=VHD_S10
   VHD_S11_Q2               patient_vignette code=VHD  sg=VHD_S11
   VHD_S12_Q2               patient_vignette code=VHD  sg=VHD_S12
   VHD_S13_Q3               patient_vignette code=VHD  sg=VHD_S13
   VHD_S14_Q2               patient_vignette code=VHD  sg=VHD_S14
   VHD_S14_Q4               

In [ ]:
# CELL 14d : batch 4
paths_b4 = do_batch_runs("batch4")
combined_b4, c_b4 = do_batch_combine("batch4")

RUN_SET=batch4: 35 questions (11 with expert corrections)
   VHD_S4_Q3                fact_based       code=VHD  sg=VHD_S4
   VHD_S6_Q1                fact_based       code=VHD  sg=VHD_S6
   VHD_S6_Q2                fact_based       code=VHD  sg=VHD_S6
   VHD_S6_Q5                fact_based       code=VHD  sg=VHD_S6
   VHD_S7_Q1                patient_vignette code=VHD  sg=VHD_S7
   VHD_S7_Q7                fact_based       code=VHD  sg=VHD_S7
   VHD_S8_Q4                patient_vignette code=VHD  sg=VHD_S8
   VHD_S8_Q5                patient_vignette code=VHD  sg=VHD_S8
   VHD_S8_Q7                patient_vignette code=VHD  sg=VHD_S8
   VHD_S9_Q5                patient_vignette code=VHD  sg=VHD_S9
   VHD_S9_Q6                patient_vignette code=VHD  sg=VHD_S9
   VHD_S9_Q7                patient_vignette code=VHD  sg=VHD_S9
   VHD_S9_Q8                patient_vignette code=VHD  sg=VHD_S9
   VHD_S11_Q1               patient_vignette code=VHD  sg=VHD_S11
   VHD_S11_Q4               pat

In [ ]:
# =========================== CELL 15 : merge -> all140 -> all160 ============
# Run only after batches 1-4 all have their combined files.
paths = sorted(glob.glob(os.path.join(RESULTS_FINAL, "combined_batch*_*.json")))
assert len(paths) == 4, f"expected 4 batch combineds, found {len(paths)}"
combined_all140, c140 = merge_combined(paths, out_tag="all140")

frozen = os.path.join(RESULTS_DIR, "combined_reviewed20.json")
assert os.path.exists(frozen), f"frozen combined not found: {frozen}"
combined_all160, c160 = merge_combined([c140, frozen], out_tag="all160")

SAVED MERGED -> /content/drive/MyDrive/medical_qa_hallucination/experiments/results/final/combined_all140_20260918_173634.json  (140 questions, 12 runs)

arm                   mean    range
coarse                10.1     6-17
atomic                11.2     7-16
atomic_gl_filter      11.2     4-17
atomic_sec_filter     11.0     5-15
SAVED MERGED -> /content/drive/MyDrive/medical_qa_hallucination/experiments/results/final/combined_all160_20260918_173634.json  (160 questions, 15 runs)

arm                   mean    range
coarse                 9.4     6-17
atomic                10.3     5-16
atomic_gl_filter      10.5     4-17
atomic_sec_filter     10.3     5-15
